In [1]:
import os
import random
import json
import numpy as np
import torch
import scanpy as sc
import pandas as pd
import celltypist
from celltypist import models
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, SequentialSampler

from scgpt.model import TransformerModel
from scgpt.tokenizer import GeneVocab
from scgpt.preprocess import Preprocessor, binning


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = 42) -> None:
    """
    Sets the random seed for reproducibility across multiple libraries.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    # tf.random.set_seed(seed)
    torch.manual_seed(seed)
    sc.settings.seed = seed
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"Random seed set as {seed}")

# Usage
set_seed(42)

print(f"✅ Research Environment Initialized on {device}")

/conda_envs/scgpt/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv
/conda_envs/scgpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch version 2.3.1+cu121 available.
JAX version 0.6.2 available.


Random seed set as 42
✅ Research Environment Initialized on cuda


## 1. Data Preparation & scGPT Loading

In [3]:
model_dir = "../../model_checkpoints/scGPT_human"
vocab = GeneVocab.from_file(f"{model_dir}/vocab.json")

In [13]:
from torch_geometric.loader import DataLoader

In [7]:
import scanpy as sc

adata = sc.read_h5ad("../../data/splits/norman_2019_raw_sparse_with_splits_original_from_GEARS.h5ad")

# scGPT expects these obs columns:
#   - 'condition': perturbation identity (e.g., "FOXA3+ctrl", "FOXA3+FOXL1")
#   - 'cell_type': (can be uniform for Norman, e.g., "K562")
#   - 'split': train/val/test assignment
# 
# Map your column:
adata.obs['split'] = adata.obs['gears_split']

In [8]:
adata

AnnData object with n_obs × n_vars = 89357 × 5045
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name', 'split', 'gears_split'
    var: 'gene_name'
    uns: 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'rank_genes_groups_cov_all', 'top_non_dropout_de_20', 'top_non_zero_de_20'
    layers: 'counts'

In [9]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)


In [10]:
sc.pp.highly_variable_genes(adata, n_top_genes=1200, subset=True)

In [11]:
hyperparameter_defaults = dict(
    seed=42,
    dataset_name="norman",
    load_model=model_dir,
    n_bins=51,           # expression binning
    epochs=15,           # typically 10-20 for Norman
    lr=1e-4,             # learning rate
    batch_size=96,       # per-GPU batch size
    layer_size=128,
    nlayers=4,           # number of transformer layers  
    nhead=4,             # attention heads
    dropout=0.2,
    amp=True,            # mixed precision
)